# BurstGPT 3: frozen vs online-adapting copy
The initial offline training **stays unchanged**. First run `colab_burstgpt3_experiments.ipynb` and export its team-results ZIP (validation-only export is sufficient). This notebook loads the same validation-selected checkpoint into two copies, then processes the selected test traffic once in chronological order.

- **Frozen:** greedy decisions, no weight updates.
- **Adapting:** identical initial weights; updates only after the environment delivers the next state and observed reward. Replay starts empty and contains only previously delivered transitions. Default exploration is zero to isolate learning.
- Both copies have independent simulations but identical native arrivals, token lengths, SLOs, hardware, observations, and reward rules. Adaptation persists from mixed traffic into peak-heavy traffic. There is no shuffle, retrospective best-checkpoint selection, or repeated pass through the stream.

**This is an online-learning experiment, not a frozen held-out test.** Do not tune adaptation settings using these results and then call them an untouched test. Choose settings on a separate development stream and use multiple independent starting seeds for final comparisons.

Use a CPU runtime. Default: a **30-minute prefix of each group** for a manageable check. Set `MAX_SECONDS_PER_GROUP=None` for the complete ranges. Full continuous replay retains many lifecycle events and can need substantial memory; use a high-RAM runtime or a larger local machine if necessary.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/xfgao210/llm-request-triage.git"
BRANCH = "burstgpt_3"  # Change to main after this branch is merged.
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    REPO = Path("/content/llm-request-triage")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "status", "--short"], check=True)
        print("Using the existing checkout. Fetch/switch explicitly if you need another revision.")
else:
    REPO = Path.cwd().resolve()
    if REPO.name == "notebooks":
        REPO = REPO.parent

required = ["experiments/online_adaptation.py", "configs/burstgpt3_ranges.json",
            "scripts/prepare_burstgpt3.py", "src/llm_triage/workload/time_ranges.py"]
assert all((REPO / name).is_file() for name in required), "Checkout lacks BurstGPT 3 changes; use the pushed burstgpt_3 branch."
os.chdir(REPO)
assert sys.version_info >= (3, 11), "Use Python 3.11 or newer."
print("Repository:", REPO)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev,rl,analysis]", "pandas>=2.0"], check=True)
sys.path.insert(0, str(REPO / "src"))

import base64
import csv
from datetime import datetime, timezone
import hashlib
import html
import importlib.metadata
import json
import shutil
import uuid
import zipfile

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display, Image, FileLink

from llm_triage.workload.burstgpt import sha256


## Inputs and adaptation settings
Put both `BurstGPT_3.csv` and the ZIP exported by the training notebook in Google Drive, then edit their paths below. Alternatively choose direct upload for either file. The original checkpoint is never overwritten.

The adapting optimizer and target network initialize fresh from checkpoint weights; the offline training checkpoint does not contain its optimizer or replay-buffer state. This is fine-tuning, not exact training continuation. Learning-rate and exploration choices below are **experimental defaults**, not validated improvements.


In [ ]:
CSV_MODE = "drive"  # @param ["drive", "upload"]
DRIVE_CSV_PATH = "/content/drive/MyDrive/BurstGPT_3.csv"  # @param {type:"string"}
LOCAL_CSV_PATH = str(REPO / "BurstGPT_3.csv")  # Used when running locally.
RUN_NAME = ""  # @param {type:"string"}
SEED = 7  # Must match a checkpoint contained in the training ZIP.
ARCHIVE_MODE = "drive"  # @param ["drive", "upload"]
TRAINING_ARCHIVE = "/content/drive/MyDrive/burstgpt3_team_results.zip"  # @param {type:"string"}
ONLINE_LEARNING_RATE = 0.0001
EPSILON = 0.0  # Exploration affects only the adapting copy; zero isolates learning.
BATCH_SIZE = 64
WARMUP = 256
UPDATE_INTERVAL = 4
REPLAY_CAPACITY = 50000
MAX_SECONDS_PER_GROUP = 1800  # None = complete ranges, not just their prefixes.
GROUPS = None  # None = both groups, in chronological order.
RUN_ONLINE_EXPERIMENT = False  # Set True deliberately after reviewing settings.
RUN_REPOSITORY_TESTS = True  # @param {type:"boolean"}

run_name = RUN_NAME or datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:6]
assert Path(run_name).name == run_name and run_name not in {".", ".."}, "Use a simple folder name."
RUN_ROOT = REPO / "results" / "runs" / "colab_burstgpt3_online" / run_name
RUN_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES = RUN_ROOT / "figures"
FIGURES.mkdir(exist_ok=True)
DATA_DIR = RUN_ROOT / "data"
ANALYSIS_DIR = RUN_ROOT / "traffic_analysis"
COMMAND_LOG = RUN_ROOT / "notebook_commands.log"
RANGE_CONFIG_PATH = RUN_ROOT / "ranges.json"
test_results = {}
generated_figures = set()
print("Run folder:", RUN_ROOT)

def run_command(args):
    command = [str(value) for value in args]
    print("Running:", " ".join(command), flush=True)
    with COMMAND_LOG.open("a", encoding="utf-8") as log:
        log.write("\n$ " + " ".join(command) + "\n")
        with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

def save_figure(fig, name):
    path = FIGURES / (name + ".png")
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    generated_figures.add(path.name)
    display(Image(filename=str(path)))
    return path

def save_config(path, value):
    if path.exists():
        assert json.loads(path.read_text()) == value, "Existing run config differs. Use a new RUN_NAME."
    else:
        path.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")


In [ ]:
if IS_COLAB:
    SOURCE_CSV = REPO / "data" / "raw" / "BurstGPT_3.csv"
    SOURCE_CSV.parent.mkdir(parents=True, exist_ok=True)
    if not SOURCE_CSV.exists():
        if CSV_MODE == "drive":
            from google.colab import drive
            drive.mount("/content/drive")
            drive_source = Path(DRIVE_CSV_PATH)
            assert drive_source.is_file(), "Set DRIVE_CSV_PATH to your BurstGPT_3.csv."
            shutil.copyfile(drive_source, SOURCE_CSV)
        elif CSV_MODE == "upload":
            from google.colab import files
            uploaded = files.upload()
            csv_files = [name for name in uploaded if Path(name).suffix.lower() == ".csv"]
            assert len(csv_files) == 1, "Upload exactly one BurstGPT CSV."
            SOURCE_CSV.write_bytes(uploaded[csv_files[0]])
            del uploaded
        else:
            raise ValueError("CSV_MODE must be drive or upload")
else:
    SOURCE_CSV = Path(LOCAL_CSV_PATH).expanduser().resolve()

assert SOURCE_CSV.is_file(), "BurstGPT_3.csv was not found."
print("CSV:", SOURCE_CSV, "Bytes:", SOURCE_CSV.stat().st_size)
if RUN_REPOSITORY_TESTS:
    run_command([sys.executable, "-m", "pytest", "-q"])


In [ ]:
if ARCHIVE_MODE == "upload" and IS_COLAB:
    from google.colab import files
    uploaded = files.upload()
    assert len(uploaded) == 1, "Upload exactly one training-results ZIP."
    TRAINING_ARCHIVE = next(iter(uploaded))
archive_path = Path(TRAINING_ARCHIVE)
assert archive_path.is_file(), "Set TRAINING_ARCHIVE to the ZIP exported by the training notebook."
TRAINING_DIR = RUN_ROOT / "source_checkpoint"
TRAINING_DIR.mkdir(exist_ok=True)
# Read only explicit root-level members; never extract arbitrary ZIP paths.
with zipfile.ZipFile(archive_path) as bundle:
    members = ["training_record.json", "dataset_manifest.json", "ranges.json", f"dqn_seed{SEED}.npz"]
    assert all(name in bundle.namelist() for name in members), "ZIP lacks the selected checkpoint or original dataset metadata."
    assert all(bundle.getinfo(name).file_size < 128 * 1024**2 for name in members), "Unexpected bundle member size."
    payload = {name: bundle.read(name) for name in members}
original_manifest_bytes = payload["dataset_manifest.json"]
original_manifest = json.loads(original_manifest_bytes)
training_record = json.loads(payload["training_record.json"])
selected = next(c for c in training_record["checkpoints"] if c["seed"] == SEED)
assert hashlib.sha256(payload[f"dqn_seed{SEED}.npz"]).hexdigest() == selected["sha256"]
assert hashlib.sha256(original_manifest_bytes).hexdigest() == training_record["manifest_sha256"]
for name, raw in {"run.json": payload["training_record.json"], f"dqn_seed{SEED}.npz": payload[f"dqn_seed{SEED}.npz"]}.items():
    destination = TRAINING_DIR / name
    if destination.exists():
        assert destination.read_bytes() == raw, "Use a new RUN_NAME for different source checkpoints."
    else:
        destination.write_bytes(raw)
range_config = json.loads(payload["ranges.json"])
assert range_config == original_manifest["range_config"]
EPISODE_SECONDS = range_config["episode_seconds"]
print("Selected offline checkpoint:", selected)
print("Original training config:", training_record["config"])


## Reconstruct exactly the same traffic
The source CSV and prepared trace hashes must match the offline run. Re-preparation on another machine changes path-dependent manifest metadata, so we verify the actual trace, ranges, windows, empty windows, and training-only estimate before restoring the **original training manifest bytes**. Its source paths remain historical provenance; the new source path is recorded separately in the export.


In [ ]:

assert sha256(SOURCE_CSV) == range_config["source_sha256"], "CSV does not match the selected BurstGPT 3 source."
save_config(RANGE_CONFIG_PATH, range_config)
display(pd.DataFrame(range_config["ranges"]))

if not (ANALYSIS_DIR / "summary.json").exists():
    run_command([sys.executable, "scripts/analyze_burstgpt_traffic.py", "--source", SOURCE_CSV,
                 "--output-dir", ANALYSIS_DIR, "--bin-seconds", 300])
analysis = json.loads((ANALYSIS_DIR / "summary.json").read_text())
assert analysis["source"]["sha256"] == range_config["source_sha256"]

if not (DATA_DIR / "processed" / "manifest.json").exists():
    run_command([sys.executable, "scripts/prepare_burstgpt3.py", "--source", SOURCE_CSV,
                 "--analysis-dir", ANALYSIS_DIR, "--ranges-config", RANGE_CONFIG_PATH, "--data-dir", DATA_DIR])
manifest = json.loads((DATA_DIR / "processed" / "manifest.json").read_text())
assert manifest["range_config"] == range_config, "Prepared ranges differ. Use a fresh run."
assert manifest["source_sha256"] == sha256(SOURCE_CSV)
assert manifest["selected_sha256"] == sha256(DATA_DIR / "processed" / "burstgpt_windows.csv")

episode_table = pd.DataFrame(manifest["windows"] + manifest.get("empty_windows", []))
episode_table["duration_s"] = episode_table["end_timestamp_s"] - episode_table["start_timestamp_s"]
episode_table["nonempty"] = episode_table["requests"] > 0
dataset_summary = episode_table.groupby(["split", "group"], sort=False).agg(
    requests=("requests", "sum"), episodes=("window", "size"),
    nonempty_episodes=("nonempty", "sum"), duration_s=("duration_s", "sum")).reset_index()
dataset_summary["no_record_episode_fraction"] = 1 - dataset_summary["nonempty_episodes"] / dataset_summary["episodes"]
dataset_summary["mean_rps_including_gaps"] = dataset_summary["requests"] / dataset_summary["duration_s"]
display(dataset_summary)
dataset_summary.to_csv(RUN_ROOT / "dataset_summary.csv", index=False)

traffic = pd.read_csv(ANALYSIS_DIR / "time_bins.csv")
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(traffic["start_timestamp_s"], traffic["mean_rps"], linewidth=0.6, color="0.5")
colors = {"train": "#2166ac", "validation": "#4daf4a", "test": "#d95f02"}
for item in range_config["ranges"]:
    ax.axvspan(item["start_timestamp_s"], item["end_timestamp_s"], alpha=0.18,
               color=colors[item["split"]], label=item["group"])
ax.ticklabel_format(axis="x", style="plain", useOffset=False)
ax.set(xlabel="Original CSV timestamp (seconds)", ylabel="Five-minute requests/s",
       title="Selected traffic ranges — natural traffic mix retained")
ax.legend()
ax.grid(alpha=0.2)
save_figure(fig, "selected_traffic_ranges")

# Content equality, not machine-specific paths, authorizes manifest restoration.
for key in ("source_sha256", "selected_sha256", "range_config", "windows", "empty_windows", "estimated_output_tokens_train_mean"):
    assert manifest[key] == original_manifest[key], f"Reconstructed dataset differs: {key}"
assert sha256(DATA_DIR / "processed/burstgpt_windows.csv") == training_record["trace_sha256"]
(DATA_DIR / "processed/manifest.json").write_bytes(original_manifest_bytes)
assert sha256(DATA_DIR / "processed/manifest.json") == training_record["manifest_sha256"]
manifest = original_manifest


## Causal replay protocol
At simulated time `t`, the copy chooses an action from its current sanitized observation. The simulator advances to the next decision and delivers only the intervening rewards/outcomes. Only then may the adapting copy add that transition to replay and update. Its next decision occurs at or after delivery time. Outcomes of requests still in flight are not available for learning. Replay may resample **past transitions**, but never future ones.

Unlike the earlier frozen notebook, this comparison uses **one continuous queue** over the selected ranges, including idle gaps. Queues and learning state do not reset every five minutes. Five-minute windows are reporting bins only. The existing time-progress observation is normalized by the total replay horizon for **both copies**, which differs from offline episodic training; compare these paired runs to each other, not directly to the earlier episodic scores.

Traffic outside the selected ranges is omitted and treated as idle, not reconstructed. GPU behavior is simulated; learner computation does not add simulated GPU latency. Training uses existing delayed environment rewards (including shaping), not future true output lengths. The final terminal bonus is delivered only at stream termination and cannot improve earlier decisions.


In [ ]:
ONLINE_OUTPUT = RUN_ROOT / "online_comparison"
online_settings = dict(seed=SEED, groups=GROUPS, epsilon=EPSILON, learning_rate=ONLINE_LEARNING_RATE,
                       batch_size=BATCH_SIZE, warmup=WARMUP, update_interval=UPDATE_INTERVAL,
                       capacity=REPLAY_CAPACITY, max_seconds_per_group=MAX_SECONDS_PER_GROUP)
display(pd.DataFrame([online_settings]))
if not RUN_ONLINE_EXPERIMENT:
    print("Not run. Set RUN_ONLINE_EXPERIMENT=True above to start the comparison.")
else:
    if ONLINE_OUTPUT.exists():
        state = json.loads((ONLINE_OUTPUT / "run.json").read_text())
        assert state["status"] == "completed", "Incomplete online runs cannot resume. Use a fresh RUN_NAME."
        assert state["settings"] == online_settings, "Settings changed. Use a fresh RUN_NAME."
        assert state["source_checkpoint_sha256"] == sha256(TRAINING_DIR / f"dqn_seed{SEED}.npz")
        assert state["manifest_sha256"] == sha256(DATA_DIR / "processed/manifest.json")
        for policy, digest in state["final_checkpoints"].items():
            assert sha256(ONLINE_OUTPUT / f"{policy}_final.npz") == digest
        print("Reusing completed comparison; not replaying or training a second time.")
    else:
        command = [sys.executable, "-m", "experiments.online_adaptation",
                   "--training-dir", TRAINING_DIR, "--data-dir", DATA_DIR, "--output", ONLINE_OUTPUT,
                   "--seed", SEED, "--epsilon", EPSILON, "--learning-rate", ONLINE_LEARNING_RATE,
                   "--batch-size", BATCH_SIZE, "--warmup", WARMUP, "--update-interval", UPDATE_INTERVAL,
                   "--capacity", REPLAY_CAPACITY]
        if GROUPS is not None:
            command += ["--groups", *GROUPS]
        if MAX_SECONDS_PER_GROUP is not None:
            command += ["--max-seconds-per-group", MAX_SECONDS_PER_GROUP]
        run_command(command)


In [ ]:
assert RUN_ONLINE_EXPERIMENT and (ONLINE_OUTPUT / "summary.csv").exists(), "Run the online comparison first."
summary = pd.read_csv(ONLINE_OUTPUT / "summary.csv")
timeline = pd.read_csv(ONLINE_OUTPUT / "timeline.csv")
learning = pd.read_csv(ONLINE_OUTPUT / "learning_summary.csv")
display(summary[["group", "policy", "arrivals", "slo_attainment", "drop_rate", "window_goodput_rps"]])
display(learning)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in zip(axes, ["slo_attainment", "drop_rate", "window_goodput_rps"],
                            ["SLO attainment", "Drop rate", "Goodput (req/s)"]):
    summary.pivot(index="group", columns="policy", values=metric).plot.bar(ax=ax, rot=0)
    ax.set_title(title)
    if metric != "window_goodput_rps":
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.set_ylim(0, 1.05 if metric == "slo_attainment" else max(0.05, float(summary[metric].max()) * 1.1))
save_figure(fig, "online_comparison")

group_names = list(timeline["group"].drop_duplicates())
fig, axes = plt.subplots(len(group_names), 1, figsize=(13, 3.5 * len(group_names)), squeeze=False)
for ax, group in zip(axes[:, 0], group_names):
    for policy, rows in timeline[timeline["group"] == group].groupby("policy"):
        rows = rows.sort_values("source_start_s")
        # Leave no-record bins missing rather than misrepresenting them as 0% SLO.
        score = rows["slo_attainment"].where(rows["arrivals"] > 0)
        ax.plot(rows["source_start_s"], score, label=policy)
    ax.set(title=f"{group}: arrival-cohort SLO attainment", xlabel="Original CSV timestamp (seconds)", ylim=(0, 1.05))
    ax.ticklabel_format(style="plain", axis="x", useOffset=False)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.legend()
save_figure(fig, "online_timeline")

audit = pd.read_csv(ONLINE_OUTPUT / "adapting_updates.csv")
if not audit.empty:
    fig, ax = plt.subplots(figsize=(13, 3))
    ax.plot(audit["source_timestamp_s"], audit["updates"])
    ax.set(xlabel="Original CSV timestamp (seconds)", ylabel="Cumulative updates", title="Learning only as simulated time advances")
    ax.ticklabel_format(style="plain", axis="x", useOffset=False)
    save_figure(fig, "online_updates")
else:
    print("No updates: selected traffic produced fewer steps than warmup. Increase the prefix length.")
delta = summary.pivot(index="group", columns="policy", values="slo_attainment")
delta["adapting_minus_frozen_percentage_points"] = 100 * (delta["adapting"] - delta["frozen"])
display(delta)
delta.to_csv(RUN_ROOT / "attainment_delta.csv")


## Interpret the result
Compare request-weighted SLO attainment, drops, and goodput **and** how performance changes over time. Learning can improve or degrade performance; neither is guaranteed. Nonzero exploration adds an exploration cost, so report it explicitly. One seed and a short prefix are a workflow check, not evidence of general improvement.

Timeline SLO uses eventual outcomes of each arrival cohort for **offline reporting only**; it is not information fed to earlier decisions. Goodput is cohort-local: on-time completions of a group's arrivals within that group's interval, not all completions flowing across its boundary. Requests arriving in empty bins are zero; no-record periods may be collection gaps. We retain delayed completions beyond the final arrival interval when computing completion SLO.


In [ ]:
provenance = dict(
    experiment="Online adaptation comparison (not frozen held-out evaluation)",
    git_revision=subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
    python=sys.version, versions={name: importlib.metadata.version(name) for name in ("numpy", "gymnasium", "pandas", "matplotlib")},
    training_archive_sha256=sha256(archive_path), local_csv_path=str(SOURCE_CSV), source_sha256=sha256(SOURCE_CSV),
    settings=online_settings,
    code_sha256={p.relative_to(REPO).as_posix(): sha256(p) for p in
                 [REPO / "pyproject.toml", *sorted((REPO / "src").rglob("*.py")),
                  *sorted((REPO / "experiments").glob("*.py")), *sorted((REPO / "scripts").glob("*.py"))]})
save_config(RUN_ROOT / "online_provenance.json", provenance)
parts = ["<!doctype html><html><head><meta charset='utf-8'><title>Online adaptation</title>",
         "<style>body{font:16px system-ui;max-width:1200px;margin:30px auto;padding:20px}img{max-width:100%}table{border-collapse:collapse}td,th{border:1px solid #ddd;padding:8px}pre{white-space:pre-wrap}</style></head><body>",
         "<h1>Frozen vs online-adapting DQN</h1><p>Online experiment, not a frozen held-out test. Identical starting weights and native traffic. Continuous queues; past-only replay; no learner wall-time in simulated GPU service.</p>",
         "<h2>Settings</h2><pre>" + html.escape(json.dumps(online_settings, indent=2)) + "</pre>",
         "<h2>Summary</h2>", summary.to_html(index=False), "<h2>Learning</h2>", learning.to_html(index=False),
         "<p>Timeline scores use eventual arrival-cohort outcomes for reporting only, never for earlier decisions. Goodput counts cohort-local completions within each interval. The default uses prefixes; see selected_ranges in run.json.</p>"]
for name in sorted(generated_figures):
    encoded = base64.b64encode((FIGURES / name).read_bytes()).decode("ascii")
    parts.append("<img alt='" + html.escape(name) + "' src='data:image/png;base64," + encoded + "'>")
parts += ["<h2>Provenance</h2><pre>", html.escape(json.dumps(provenance, indent=2)), "</pre></body></html>"]
REPORT = RUN_ROOT / "online_report.html"
REPORT.write_text("\n".join(parts), encoding="utf-8")
ARCHIVE = RUN_ROOT / (run_name + "_online_results.zip")
# Explicit allowlist: no raw CSV or prepared request data.
members = [REPORT, RUN_ROOT / "online_provenance.json", RUN_ROOT / "attainment_delta.csv", RANGE_CONFIG_PATH]
members += [ONLINE_OUTPUT / name for name in ("summary.csv", "timeline.csv", "learning_summary.csv", "run.json",
            "frozen_updates.csv", "adapting_updates.csv", "frozen_final.npz", "adapting_final.npz")]
members += [TRAINING_DIR / "run.json", TRAINING_DIR / f"dqn_seed{SEED}.npz",
            DATA_DIR / "processed/manifest.json", COMMAND_LOG]
members += [FIGURES / name for name in sorted(generated_figures)]
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as bundle:
    for p in members:
        bundle.write(p, p.relative_to(RUN_ROOT).as_posix())
display(FileLink(str(REPORT)))
display(FileLink(str(ARCHIVE)))
if IS_COLAB:
    from google.colab import files
    files.download(str(ARCHIVE))
print("Also save a notebook copy with outputs in Drive to share the executed analysis.")
